# 01 — Source Domain: Train & Evaluate

Trains the V-L retriever on the source domain (natural photos) and
establishes the in-domain upper bound the rest of the study compares
against. This mirrors `experiments/baseline/train.py` but interactively,
for inspecting intermediate embeddings/loss curves.

In [ ]:
import sys
sys.path.append("..")

import torch
import yaml
from torch.utils.data import DataLoader

from src.datasets import build_source_dataset
from src.models import VisionLanguageRetriever
from src.adaptation import clip_contrastive_loss
from src.evaluation import evaluate

device = "cuda" if torch.cuda.is_available() else "cpu"
device

## Load configs and build the model

In [ ]:
with open("../configs/model.yaml") as f:
    model_cfg = yaml.safe_load(f)
with open("../configs/data.yaml") as f:
    data_cfg = yaml.safe_load(f)
with open("../configs/source.yaml") as f:
    exp_cfg = yaml.safe_load(f)

model = VisionLanguageRetriever.from_config(model_cfg)
tokenizer = model.tokenizer
model.to(device);

## Build source dataloaders

Requires `data/source/splits.json` — run `data/scripts/prepare_source.py` first.

In [ ]:
train_ds = build_source_dataset(data_cfg, split="train", tokenizer=tokenizer)
val_ds = build_source_dataset(data_cfg, split="val", tokenizer=tokenizer)

dl_cfg = data_cfg["dataloader"]
train_loader = DataLoader(train_ds, batch_size=dl_cfg["batch_size"], shuffle=True, num_workers=dl_cfg["num_workers"])
val_loader = DataLoader(val_ds, batch_size=dl_cfg["batch_size"], shuffle=False, num_workers=dl_cfg["num_workers"])

len(train_ds), len(val_ds)

## Train loop (short interactive version — full run via `experiments/baseline/train.py`)

In [ ]:
optim = torch.optim.AdamW(model.parameters(), lr=exp_cfg["train"]["lr"], weight_decay=exp_cfg["train"]["weight_decay"])
loss_history = []

n_epochs = 2  # keep short for interactive exploration; full training uses configs/source.yaml's epoch count
for epoch in range(n_epochs):
    model.train()
    for batch in train_loader:
        images = batch["image"].to(device)
        tokens = batch["text"].to(device)
        img_e, txt_e = model(images, tokens)
        loss = clip_contrastive_loss(img_e, txt_e, temperature=exp_cfg["train"]["temperature"])
        optim.zero_grad(); loss.backward(); optim.step()
        loss_history.append(loss.item())
    print(f"epoch {epoch+1}: last batch loss = {loss_history[-1]:.4f}")

In [ ]:
import matplotlib.pyplot as plt
plt.plot(loss_history)
plt.xlabel("step"); plt.ylabel("contrastive loss"); plt.title("Source-domain training loss")
plt.show()

## Held-out source evaluation (the upper bound row in the README results table)

In [ ]:
result = evaluate(model, val_loader, device=device, out_file="../results/tables/source_eval.json")
result["metrics"]

## Next

Save this checkpoint to `experiments/baseline/checkpoints/best.pt` (or run
the full `experiments/baseline/train.py` script for the complete training
schedule), then continue to `02_target_domain.ipynb`.